# Part 6: Competition — Best AI Poet

You've trained a GPT on Shakespeare. Now put what you've learned to work: train the best poetry-generating model you can and submit your best poem.

## The Challenge

Train a model that generates the best poem. You choose the dataset, the model size, the tokenizer, the training strategy — everything is fair game as long as:

1. You trained the model yourself (no pretrained weights)
2. The poem was generated by the model, not written or edited by you
3. You can prove it — submit your checkpoint and the exact generation command

## Submission

Submit three things:
1. **The poem** — your model's best output
2. **The checkpoint** — so we can reproduce it  
3. **The generation settings** — exact prompt, temperature, top_k, and seed

## Judging Criteria

- **Coherence** — does it make sense as a poem?
- **Creativity** — interesting imagery, unexpected phrasing
- **Structure** — does it have rhythm, line breaks, stanzas?
- **Emotion** — does it evoke something?

## Setup

In [ ]:
!pip install -q torch tqdm matplotlib tiktoken

import torch
import torch.nn as nn
from dataclasses import dataclass
import math, json, os, urllib.request
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

os.makedirs("data", exist_ok=True)
print("Ready.")

## The Model (with optional Dropout)

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.0   # set to 0.1 to reduce overfitting

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.attn_drop = nn.Dropout(config.dropout)
        self.resid_drop = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        y = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, is_causal=True, dropout_p=self.attn_drop.p if self.training else 0.0
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.drop = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.drop(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.transformer.drop(
            self.transformer.wte(idx) + self.transformer.wpe(pos)
        )
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)
            )
        return logits, loss

print("Model with optional dropout ready.")

## Training Pipeline

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def load_data_char(filepath, block_size, batch_size, device):
    """Character-level tokenizer"""
    with open(filepath, "r") as f:
        text = f.read()
    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {c: i for i, c in enumerate(chars)}
    itos = {i: c for c, i in stoi.items()}
    tokens = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    print(f"Dataset: {len(tokens):,} chars, vocab size: {vocab_size}")
    def get_batch(split):
        ix = torch.randint(len(split) - block_size - 1, (batch_size,))
        x = torch.stack([split[i:i + block_size] for i in ix]).to(device)
        y = torch.stack([split[i + 1:i + block_size + 1] for i in ix]).to(device)
        return x, y
    n = int(0.9 * len(tokens))
    return lambda: get_batch(tokens[:n]), lambda: get_batch(tokens[n:]), vocab_size, stoi, itos

def load_data_bpe(filepath, block_size, batch_size, device):
    """BPE tokenizer (GPT-2) — use for datasets larger than ~5MB"""
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    with open(filepath, "r") as f:
        text = f.read()
    token_ids = enc.encode(text)
    vocab_size = enc.n_vocab
    stoi = None  # BPE uses tiktoken directly
    itos = None
    tokens = torch.tensor(token_ids, dtype=torch.long)
    print(f"Dataset: {len(tokens):,} BPE tokens, vocab size: {vocab_size}")
    def get_batch(split):
        ix = torch.randint(len(split) - block_size - 1, (batch_size,))
        x = torch.stack([split[i:i + block_size] for i in ix]).to(device)
        y = torch.stack([split[i + 1:i + block_size + 1] for i in ix]).to(device)
        return x, y
    n = int(0.9 * len(tokens))
    return lambda: get_batch(tokens[:n]), lambda: get_batch(tokens[n:]), vocab_size, enc, enc

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def generate(model, prompt, stoi, itos, max_new_tokens=200, temperature=0.8, top_k=40):
    device = next(model.parameters()).device
    # Support both character-level dicts and tiktoken encoders
    if hasattr(stoi, 'encode'):
        tokens = stoi.encode(prompt)
        decode_fn = lambda ids: itos.decode(ids)
    else:
        tokens = [stoi[c] for c in prompt if c in stoi]
        decode_fn = lambda ids: "".join([itos[i] for i in ids])
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k > 0:
            values, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < values[:, -1:]] = float("-inf")
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return decode_fn(idx[0].tolist())

def train(data_path, max_steps=5000, batch_size=64,
          n_layer=6, n_head=6, n_embd=384, block_size=256,
          dropout=0.0, max_lr=1e-3, use_bpe=False,
          checkpoint_name="checkpoint_final"):
    device = get_device()
    print(f"Device: {device}")

    loader = load_data_bpe if use_bpe else load_data_char
    get_train_batch, get_val_batch, vocab_size, stoi, itos = loader(
        data_path, block_size, batch_size, device
    )
    config = GPTConfig(vocab_size=vocab_size, block_size=block_size,
                       n_layer=n_layer, n_head=n_head, n_embd=n_embd, dropout=dropout)
    model = GPT(config).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_layer}L/{n_head}H/{n_embd}D — {n_params/1e6:.1f}M params")
    if dropout > 0:
        print(f"Dropout: {dropout}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=0.01)
    min_lr = max_lr * 0.1
    warmup_steps = 100
    loss_log = {"steps": [], "train": [], "val_steps": [], "val": []}

    best_val_loss = float("inf")
    best_ckpt_path = f"{checkpoint_name}_best.pt"

    pbar = tqdm(range(max_steps), desc="Training")
    for step in pbar:
        if step % 100 == 0:
            model.eval()
            with torch.no_grad():
                val_loss = sum(
                    model(*get_val_batch())[1].item() for _ in range(20)
                ) / 20
                tqdm.write(f"Step {step:5d} | val loss: {val_loss:.4f}")
                loss_log["val_steps"].append(step)
                loss_log["val"].append(val_loss)
                # Early stopping: save best checkpoint
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    torch.save({
                        "step": step, "model_state_dict": model.state_dict(),
                        "config": config, "stoi": stoi, "itos": itos,
                        "val_loss": val_loss,
                    }, best_ckpt_path)
            model.train()

        lr = get_lr(step, warmup_steps, max_steps, max_lr, min_lr)
        for g in optimizer.param_groups:
            g["lr"] = lr

        x, y = get_train_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")
        loss_log["steps"].append(step)
        loss_log["train"].append(loss.item())

        if step > 0 and step % 500 == 0:
            model.eval()
            sample = generate(model, "The morning", stoi, itos,
                              max_new_tokens=80, temperature=0.8)
            tqdm.write(f"\n--- Step {step} ---\n{sample}\n---\n")
            model.train()

    torch.save({
        "step": max_steps, "model_state_dict": model.state_dict(),
        "config": config, "stoi": stoi, "itos": itos,
    }, f"{checkpoint_name}.pt")

    log_path = f"{checkpoint_name}_loss_log.json"
    with open(log_path, "w") as f:
        json.dump(loss_log, f)

    print(f"\nDone. Best val loss: {best_val_loss:.4f} (saved to {best_ckpt_path})")
    return model, stoi, itos, loss_log

print("Training pipeline ready.")

## Lever 1: Better Data

Shakespeare worked, but it's just one style. Find or build a better poetry dataset.

Quality matters more than quantity — 1MB of carefully curated poetry will beat 10MB of random text with poems mixed in.

In [ ]:
# Option 1: Shakespeare (baseline)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "data/shakespeare.txt"
)
print(f"Shakespeare: {os.path.getsize('data/shakespeare.txt') / 1e6:.2f} MB")

# Option 2: Gutenberg sonnets (Shakespeare's sonnets only)
# urllib.request.urlretrieve(
#     "https://www.gutenberg.org/cache/epub/1041/pg1041.txt",
#     "data/sonnets.txt"
# )

# Option 3: Combine multiple sources — append files together
# with open("data/combined.txt", "w") as out:
#     for fname in ["data/shakespeare.txt", "data/sonnets.txt"]:
#         with open(fname) as f:
#             out.write(f.read())
#             out.write("\n\n")
# print(f"Combined: {os.path.getsize('data/combined.txt') / 1e6:.2f} MB")

## Lever 2: Bigger Model

With more data, you can justify a larger model without overfitting immediately.

| Config | Params | n_layer | n_head | n_embd |
|--------|--------|---------|--------|--------|
| Workshop default | ~10M | 6 | 6 | 384 |
| Larger | ~25M | 8 | 8 | 512 |
| GPT-2 Small | ~85M | 12 | 12 | 768 |

Remember: a bigger model on the same data just memorizes faster. Scale the model with the data.

In [ ]:
# Compare parameter counts
sizes = [
    ("Medium (default)", 6, 6, 384),
    ("Larger",           8, 8, 512),
    ("GPT-2 Small",     12, 12, 768),
]

print(f"{'Name':<20} {'n_layer':>8} {'n_head':>8} {'n_embd':>8} {'Params':>12}")
print("-" * 60)
for name, nl, nh, ne in sizes:
    cfg = GPTConfig(n_layer=nl, n_head=nh, n_embd=ne)
    m = GPT(cfg)
    p = sum(p.numel() for p in m.parameters())
    print(f"{name:<20} {nl:>8} {nh:>8} {ne:>8} {p/1e6:>11.1f}M")

## Lever 3: Better Tokenizer

Character-level worked for Shakespeare because the dataset was small. With a larger poetry dataset (>5MB), consider BPE:

- **BPE with tiktoken**: GPT-2's tokenizer — sequences are ~3× shorter, so each batch covers more text
- **Trade-off**: vocab_size=50,257 means a much larger embedding table and output layer

In [ ]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")
with open("data/shakespeare.txt") as f:
    text = f.read()

char_tokens = len(text)
bpe_tokens = len(enc.encode(text))

print(f"Shakespeare character tokens:  {char_tokens:,}")
print(f"Shakespeare BPE tokens (GPT-2): {bpe_tokens:,}")
print(f"Compression: {char_tokens/bpe_tokens:.1f}× longer with char-level")
print()
print("For a 5MB poetry dataset:")
data_5mb = 5_000_000
print(f"  Character tokens: ~{data_5mb:,}")
print(f"  BPE tokens:       ~{data_5mb//3:,}")
print()
print("With block_size=256 and batch_size=64:")
print(f"  Tokens per step: {256*64:,}")
print(f"  Steps to cover 5MB (char): ~{data_5mb//(256*64):,}")
print(f"  Steps to cover 5MB (BPE):  ~{(data_5mb//3)//(256*64):,}")

## Lever 4: Training Tweaks

- **Longer context** (`block_size=512` or `1024`): lets the model capture full stanzas and rhyme schemes
- **Dropout** (`dropout=0.1`): reduces overfitting by randomly zeroing activations during training
- **Early stopping**: the training loop above saves the checkpoint with the lowest val loss automatically
- **Learning rate tuning**: try `max_lr=3e-4` (conservative) vs `3e-3` (aggressive)

## Run Your Competition Training

Modify the cell below with your chosen strategy and train your best model.

In [ ]:
# === YOUR COMPETITION TRAINING ===
# Tune these parameters to build your best model!

model, stoi, itos, loss_log = train(
    data_path="data/shakespeare.txt",
    max_steps=5000,
    batch_size=64,
    n_layer=6,
    n_head=6,
    n_embd=384,
    block_size=256,
    dropout=0.0,        # try 0.1 to reduce overfitting
    max_lr=1e-3,
    use_bpe=False,      # set True for large datasets (>5MB)
    checkpoint_name="competition"
)

## Generate Your Best Poem

Load the **best** checkpoint (lowest val loss), not the final one. Then generate many samples and cherry-pick.

In [ ]:
# Load the best checkpoint (saved during training)
best_ckpt = torch.load("competition_best.pt", weights_only=False, map_location="cpu")
best_config = best_ckpt["config"]
best_stoi = best_ckpt["stoi"]
best_itos = best_ckpt["itos"]

best_model = GPT(best_config)
best_model.load_state_dict(best_ckpt["model_state_dict"])
best_model.eval()

print(f"Loaded best checkpoint from step {best_ckpt['step']}")
print(f"Val loss at checkpoint: {best_ckpt.get('val_loss', 'N/A')}")

In [ ]:
# Try different prompts and settings — generate many, keep the best
PROMPT = "The morning sun"  # change this!
TEMPERATURE = 0.8
TOP_K = 40
N_SAMPLES = 10

print(f"Generating {N_SAMPLES} samples with prompt: '{PROMPT}'\n")

for i in range(N_SAMPLES):
    output = generate(best_model, PROMPT, best_stoi, best_itos,
                      max_new_tokens=200, temperature=TEMPERATURE, top_k=TOP_K)
    print(f"--- Sample {i+1} ---")
    print(output)
    print()

## Reproduce Your Submission

Once you've found your best poem, record the exact seed so it's reproducible.

In [ ]:
# === FILL IN YOUR SUBMISSION DETAILS ===
SUBMISSION_PROMPT = "The morning sun"   # your best prompt
SUBMISSION_TEMPERATURE = 0.8
SUBMISSION_TOP_K = 40
SUBMISSION_SEED = 42                    # the seed that produced your best poem

torch.manual_seed(SUBMISSION_SEED)
best_poem = generate(
    best_model, SUBMISSION_PROMPT, best_stoi, best_itos,
    max_new_tokens=200,
    temperature=SUBMISSION_TEMPERATURE,
    top_k=SUBMISSION_TOP_K
)

print("=" * 60)
print("YOUR SUBMISSION")
print("=" * 60)
print(best_poem)
print("=" * 60)
print()
print("Generation settings (include in your submission):")
print(f"  Prompt:      {repr(SUBMISSION_PROMPT)}")
print(f"  Temperature: {SUBMISSION_TEMPERATURE}")
print(f"  Top-k:       {SUBMISSION_TOP_K}")
print(f"  Seed:        {SUBMISSION_SEED}")
print(f"  Checkpoint:  competition_best.pt")